In [3]:
import os
import sys
import torch
import numpy as np
import cv2
from tqdm import tqdm

In [4]:
# source code
SRC_PATH = "/kaggle/input/datasets/priyanshusghosh/source/src/"
sys.path.append(SRC_PATH)

# datasets
RAW_DIR = "/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images/cell_images"
PROCESSED_DIR = "/kaggle/input/datasets/priyanshusghosh/processed"

# splits
SPLIT_DIR = "/kaggle/input/datasets/priyanshusghosh/splits"

# output
MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
from data.loader import load_image_paths, read_image
from models.efficientnet import get_efficientnet
from train.train import train_model
from train.evaluate import evaluate
from utils.metrics import compute_metrics
from utils.latency import measure_latency

In [6]:
data = load_image_paths(RAW_DIR)

train_idx = np.load(os.path.join(SPLIT_DIR, "train_idx.npy"))
test_idx  = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

train_data = [data[i] for i in train_idx]
test_data  = [data[i] for i in test_idx]

In [7]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, data, processed=False):
        self.data = data
        self.processed = processed

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]

        if self.processed:
            # map to processed path
            class_name = "Parasitized" if label == 0 else "Uninfected"
            filename = os.path.basename(path)
            path = os.path.join(PROCESSED_DIR, class_name, filename)
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = read_image(path)

        img = cv2.resize(img, (224, 224))
        img = img / 255.0

        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        label = torch.tensor(label)

        return img, label

In [8]:
batch_size = 32

train_raw = torch.utils.data.DataLoader(
    SimpleDataset(train_data, processed=False),
    batch_size=batch_size,
    shuffle=True
)

test_raw = torch.utils.data.DataLoader(
    SimpleDataset(test_data, processed=False),
    batch_size=batch_size,
    shuffle=False
)

train_pre = torch.utils.data.DataLoader(
    SimpleDataset(train_data, processed=True),
    batch_size=batch_size,
    shuffle=True
)

test_pre = torch.utils.data.DataLoader(
    SimpleDataset(test_data, processed=True),
    batch_size=batch_size,
    shuffle=False
)

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_raw = get_efficientnet()
model_raw, hist_raw = train_model(model_raw, train_raw, test_raw, epochs=5, device=device)

torch.save(model_raw.state_dict(), os.path.join(MODEL_DIR, "efficientnet_raw.pth"))

model_pre = get_efficientnet()
model_pre, hist_pre = train_model(model_pre, train_pre, test_pre, epochs=5, device=device)

torch.save(model_pre.state_dict(), os.path.join(MODEL_DIR, "efficientnet_pre.pth"))

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 148MB/s]
100%|██████████| 689/689 [03:05<00:00,  3.71it/s]


In [10]:
def get_preds(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            p = torch.argmax(out, dim=1).cpu().numpy()

            preds.extend(p)
            labels.extend(y.numpy())

    return np.array(labels), np.array(preds)

In [11]:
# RAW
y_true_raw, y_pred_raw = get_preds(model_raw, test_raw)
metrics_raw = compute_metrics(y_true_raw, y_pred_raw)

# PRE
y_true_pre, y_pred_pre = get_preds(model_pre, test_pre)
metrics_pre = compute_metrics(y_true_pre, y_pred_pre)

print("RAW:", metrics_raw)
print("PRE:", metrics_pre)

RAW: {'accuracy': 0.9619013062409288, 'f1': 0.9628055260361318, 'precision': 0.9404844290657439, 'recall': 0.9862119013062409, 'confusion_matrix': array([[2584,  172],
       [  38, 2718]])}
PRE: {'accuracy': 0.9517416545718432, 'f1': 0.9532184312346114, 'precision': 0.9249146757679181, 'recall': 0.9833091436865021, 'confusion_matrix': array([[2536,  220],
       [  46, 2710]])}


In [12]:
lat_raw = measure_latency(model_raw, test_raw, device)
lat_pre = measure_latency(model_pre, test_pre, device)

print("Latency RAW:", lat_raw)
print("Latency PRE:", lat_pre)

Latency RAW: 2.8890334123796926
Latency PRE: 2.9041822918613693


In [13]:
def get_size(path):
    return os.path.getsize(path) / (1024 * 1024)

size_raw = get_size(os.path.join(MODEL_DIR, "efficientnet_raw.pth"))
size_pre = get_size(os.path.join(MODEL_DIR, "efficientnet_pre.pth"))

In [14]:
import csv

csv_path = "/kaggle/working/metrics.csv"

rows = [
    ["EfficientNet_Raw", metrics_raw["accuracy"], metrics_raw["f1"],
     metrics_raw["precision"], metrics_raw["recall"], lat_raw, size_raw],

    ["EfficientNet_Pre", metrics_pre["accuracy"], metrics_pre["f1"],
     metrics_pre["precision"], metrics_pre["recall"], lat_pre, size_pre],
]

header = ["Model", "Accuracy", "F1", "Precision", "Recall", "Latency(ms)", "Size(MB)"]

if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
else:
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(rows)

In [16]:
model = get_efficientnet()
model.load_state_dict(torch.load("/kaggle/working/models/efficientnet_pre.pth"))
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [17]:
from optimize.quantize import quantize_model

model_q = quantize_model(model)

In [18]:
def get_preds_cpu(model, loader):
    model.to("cpu")
    model.eval()

    preds, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to("cpu")   # force CPU
            out = model(x)
            p = torch.argmax(out, dim=1).numpy()

            preds.extend(p)
            labels.extend(y.numpy())

    return np.array(labels), np.array(preds)

In [19]:
y_true_q, y_pred_q = get_preds_cpu(model_q, test_pre)
metrics_q = compute_metrics(y_true_q, y_pred_q)

print(metrics_q)

{'accuracy': 0.9517416545718432, 'f1': 0.9532184312346114, 'precision': 0.9249146757679181, 'recall': 0.9833091436865021, 'confusion_matrix': array([[2536,  220],
       [  46, 2710]])}


In [20]:
lat_q = measure_latency(model_q, test_pre, device="cpu")  # IMPORTANT: CPU
print(lat_q) 

46.60561728373668


In [21]:
torch.save(model_q.state_dict(), "/kaggle/working/models/efficientnet_quant.pth")

size_q = os.path.getsize("/kaggle/working/models/efficientnet_quant.pth") / (1024*1024)
print(size_q)

15.574458122253418


In [22]:
csv_path = "/kaggle/working/metrics.csv"

row = [
    "EfficientNet_Quant",
    metrics_q["accuracy"],
    metrics_q["f1"],
    metrics_q["precision"],
    metrics_q["recall"],
    lat_q,
    size_q
]

with open(csv_path, "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(row)

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns

def save_cm(cm, name):
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(name)
    plt.savefig(f"/kaggle/working/{name}.png")
    plt.close()

save_cm(metrics_raw["confusion_matrix"], "efficientnet_raw_cm")
save_cm(metrics_pre["confusion_matrix"], "efficientnet_pre_cm")
save_cm(metrics_q["confusion_matrix"], "efficientnet_quant_cm")

In [24]:
import matplotlib.pyplot as plt

# Loss curve
plt.plot(hist_pre["train_loss"], label="train")
plt.plot(hist_pre["val_loss"], label="val")
plt.legend()
plt.title("EfficientNet Preprocessed - Loss")
plt.savefig("/kaggle/working/efficientnet_pre_loss.png")
plt.close()

# Accuracy curve
plt.plot(hist_pre["val_acc"], label="val_acc")
plt.legend()
plt.title("EfficientNet Preprocessed - Accuracy")
plt.savefig("/kaggle/working/efficientnet_pre_acc.png")
plt.close()

In [25]:
import matplotlib.pyplot as plt

# Loss curve
plt.plot(hist_raw["train_loss"], label="train")
plt.plot(hist_raw["val_loss"], label="val")
plt.legend()
plt.title("EfficientNet Raw - Loss")
plt.savefig("/kaggle/working/efficientnet_raw_loss.png")
plt.close()

# Accuracy curve
plt.plot(hist_raw["val_acc"], label="val_acc")
plt.legend()
plt.title("EfficientNet Raw - Accuracy")
plt.savefig("/kaggle/working/efficientnet_raw_acc.png")
plt.close()